#  1 - Importing Necessary Libraries

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 2 - Data Cleaning and Preparation

In [22]:
column_names = ['fLength', 'fWidth', 'fSize', 'fConc', 'fConc1',
                'fAsym', 'fM3Long', 'fM3Trans', 'fAlpha', 'fDist', 'class']

data = pd.read_csv('magic04.data',names = column_names)

In [8]:
print(data.isnull().sum())

fLength     0
fWidth      0
fSize       0
fConc       0
fConc1      0
fAsym       0
fM3Long     0
fM3Trans    0
fAlpha      0
fDist       0
class       0
dtype: int64


## 2.1 Convert class labels to numeric

In [10]:
class_mapping = {'g': 1, 'h': 0} 
data['class'] = data['class'].map(class_mapping)

## 2.2 Data Balancing

In [12]:
class_0 = data[data['class'] == 0] 
class_1 = data[data['class'] == 1]

# Undersample the majority class (class 1) to match the size of the minority class (class 0)
class_1_undersampled = class_1.sample(len(class_0), random_state=42)

# Combine the minority class and the undersampled majority class
balanced_df = pd.concat([class_0, class_1_undersampled], axis=0)

# Shuffle the balanced dataset
balanced_df = balanced_df.sample(frac=1, random_state=42).reset_index(drop=True)

print("\nBalanced dataset shape:", balanced_df.shape)
print("\nBalanced class distribution (Counts):")
print(balanced_df['class'].value_counts())


Balanced dataset shape: (13376, 11)

Balanced class distribution (Counts):
class
1    6688
0    6688
Name: count, dtype: int64


## 2.3 Data Splitting

In [14]:
X = balanced_df.drop(columns=['class'])
y = balanced_df['class']

# Split into 70% training and 30% testing
# Using stratify ensures the class proportion is maintained in train/test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print(f"\nX_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")


X_train shape: (9363, 10)
y_train shape: (9363,)
X_test shape: (4013, 10)
y_test shape: (4013,)


## 2.4 Feature Scaling

In [16]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\nX_train_scaled shape: {X_train_scaled.shape}")
print(f"X_test_scaled shape: {X_test_scaled.shape}")


X_train_scaled shape: (9363, 10)
X_test_scaled shape: (4013, 10)


# Naive Bayes Classification Model

This code cell implements a Gaussian Naive Bayes classifier for a machine learning classification task. Here's what the code does:

1. **Imports necessary libraries**:
   - `GaussianNB` from scikit-learn's naive_bayes module
   - Classification evaluation metrics from scikit-learn

2. **Creates and trains the model**:
   - Initializes a Gaussian Naive Bayes classifier
   - Fits the model using the scaled training data (`X_train_scaled` and `y_train`)

3. **Makes predictions**:
   - Uses the trained model to predict classes for the scaled test data

4. **Evaluates model performance**:
   - Generates a classification report showing precision, recall, f1-score, and support
   - Creates a confusion matrix to visualize prediction errors
   - Prints both evaluation metrics to assess model performance

In [36]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report,confusion_matrix

model=GaussianNB()
model.fit(X_train_scaled,y_train)

y_predict=model.predict(X_test_scaled)

report=classification_report(y_test,y_predict)
matrix=confusion_matrix(y_test,y_predict)
print("Classification Report:\n",report)
print("Confusion Matrix:\n",matrix)

Classification Report:
               precision    recall  f1-score   support

           0       0.79      0.39      0.52      2007
           1       0.59      0.90      0.72      2006

    accuracy                           0.64      4013
   macro avg       0.69      0.64      0.62      4013
weighted avg       0.69      0.64      0.62      4013

Confusion Matrix:
 [[ 781 1226]
 [ 205 1801]]
